# 02 - Speaker-Listener: when your teammate keeps changing

**Setting: cooperative, one common reward.**
* The **speaker** sees the colour of the goal landmark but cannot move. Its action is a message of 3 numbers.
* The **listener** moves, sees the landmarks and the message, but not the goal.
* Both get the same reward every step: **minus the squared distance listener -> goal**.

The message has no built-in meaning: the two have to invent one *while both are learning*.
All runs here are pre-trained; only the last cell trains (about a minute).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import *          # load_run, describe_networks, evolution_frames, plot_curves, ...
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90

## The Markov game

Two agents, one **common reward** - a cooperative, partially observable Markov game. One episode = **25 steps**; both agents act at the same time.

| | speaker | listener |
|---|---|---|
| **observation** | 3 numbers: the **colour of the goal landmark** (RGB) | 11 numbers: own **velocity** (2), position of each of the 3 **landmarks relative to itself** (3 x 2), the speaker's **message** from the last step (3) |
| **does not see** | any position, the listener, the landmarks | which landmark is the goal |
| **action** | **message**: 3 continuous numbers in [0, 1], delivered to the listener's observation at the next step | **movement**: 5 numbers in [0, 1] = [no-op, push left, push right, push down, push up]; the net push is (right - left, up - down) |
| **can move?** | no | yes (velocity is damped, so it slows down when it stops pushing) |

**Reward** (identical for both, every step): **minus the squared distance between the listener and the goal landmark**.
The best possible return is 0 (listener sitting on the goal for all 25 steps).

Neither agent can solve the task alone: the speaker knows *where to go* but cannot move, the listener can move but does not know *where*.
The only link between them is the message - and its meaning is not given, it has to be learned.

The same, read directly from the environment (with one actual observation of each agent):

In [ ]:
from marl.render import make_mpe, describe
env = make_mpe("speaker_listener")
describe(env)
obs, _ = env.reset(seed=0)
print("\nspeaker  observation (goal colour):", obs["speaker_0"].round(2))
o = obs["listener_0"]
print("listener observation:  velocity", o[0:2].round(2), "| landmarks relative to me",
      [list(o[i:i + 2].round(2)) for i in (2, 4, 6)], "| message", o[8:11].round(2))
env.close()

## Step 1 - the obvious approach: each agent learns on its own (IDDPG)

Actions are continuous, so each agent runs its own **DDPG**: a deterministic actor (plus exploration noise), a critic that
scores actions, a replay buffer and slowly-updated target networks. "Independent" means each agent's critic only sees **its own**
observation and action - the other agent is treated as part of the environment.

In [ ]:
iddpg = load_run("sl_iddpg")
describe_networks(iddpg)

In [ ]:
plot_curves([iddpg], "goal_dist", title="IDDPG: listener's final distance to the goal (lower is better)");

### Watch it learn
The same episode replayed with the policy from later and later in training. The speaker's message is printed at the bottom.

In [ ]:
show_video(evolution_frames(iddpg, k=5), fps=10)

### What goes wrong
* The listener **wanders between landmarks** for a long time and rarely commits to the right one.
* For a long stretch the speaker sends **almost the same message for every goal** (see the "language" plot further down) - there is nothing to decode yet.
* Learning is slow and noisy.

**Why:** each agent's critic learns "how good was my action?" while the *other* agent keeps changing. When the speaker changes what a
message means, the listener's experience - and its critic - become outdated. From each agent's point of view the world is
**non-stationary**: the same situation and action lead to different outcomes over time.

## Step 2 - MADDPG: let the critics see everybody

One change: during training, each agent's critic sees **all** observations and **both** agents' actions, *Q_i(s, a_speaker, a_listener)*.
The other agent's behaviour is now an *input* rather than a hidden, changing part of the world.
The actors are unchanged and still act on their own observation only (**centralised training, decentralised execution**).

In [ ]:
maddpg = load_run("sl_maddpg")
describe_networks(maddpg)        # compare the critic inputs with IDDPG above

In [ ]:
show_video(evolution_frames(maddpg, k=5), fps=10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([iddpg, maddpg], "goal_dist", ax=ax[0], title="final distance to the goal (lower is better)")
plot_curves([iddpg, maddpg], "success", ax=ax[1], title="episodes ending with the listener on the goal")
plt.tight_layout()

MADDPG learns the task several times faster (we checked three random seeds). Note it is not perfectly stable either -
its curve has dips. The next plots show what happens in those moments.

## Looking inside: the speaker's "language"
The speaker only ever sees the goal colour, so its whole policy is **three messages** (one per goal).
* **separation** - how different the messages for different goals are (0 = the listener cannot possibly tell goals apart)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5.5, 3.4))
for run in (iddpg, maddpg):
    steps, msgs = speaker_messages(run)
    ax.plot(steps / 1e6, message_separation(msgs), marker="o", ms=3, label=run["config"]["label"])
ax.set_title("separation between goal messages");
ax.set_xlabel("environment steps (millions)"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout()

## Takeaways
* Independent learners treat each other as part of a changing world -> slow, noisy learning (**non-stationarity**).
* A critic that sees everybody's actions (MADDPG) makes the learning problem stationary from the critic's point of view and learns much faster.
* Even then the agents keep changing *together* - a teammate's "language" can shift and leave you behind.

## Try it yourself (about a minute on a laptop CPU)
A short MADDPG run from scratch. Change `critic="local"` to train IDDPG instead.

In [ ]:
import tempfile, torch
from marl.common.runner import Run
from marl.algos.ddpg import DDPGConfig, train
torch.set_num_threads(2)

steps = 100_000          # ~1 min on a laptop CPU
cfg = DDPGConfig(critic="central", warmup_steps=10_000)
run = Run("toy", "speaker_listener", {}, {}, out_root=tempfile.mkdtemp(), eval_episodes=16)
train(run, "speaker_listener", {}, cfg, total_steps=steps, eval_every=20_000, label="my MADDPG")
toy = load_run(run.dir)
plot_curves([toy], "goal_dist", title="my run: goal distance");

In [ ]:
show_video(video(load_policy(toy["ckpts"][-1]), "speaker_listener", {}, title="my MADDPG"))